# Download All Documents from Azure Blob Storage Container
This notebook downloads all files from the 'suzlon-documents' container to a local directory.

In [1]:
import os
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
from pathlib import Path
from tqdm import tqdm
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [2]:
# Load environment variables
load_dotenv()

# Configuration from .env file
CONFIG = {
    "storage_connection_string": os.getenv("AZURE_STORAGE_CONNECTION_STRING"),
    "container_name": "suzlon-documents",
    "download_directory": "./downloaded_documents"
}

print("Configuration loaded successfully!")
print(f"Container: {CONFIG['container_name']}")
print(f"Download Directory: {CONFIG['download_directory']}")

Configuration loaded successfully!
Container: suzlon-documents
Download Directory: ./downloaded_documents


In [3]:
# Initialize Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(
    CONFIG["storage_connection_string"]
)

# Get container client
container_client = blob_service_client.get_container_client(CONFIG["container_name"])

print(f"Connected to container: {CONFIG['container_name']}")

Connected to container: suzlon-documents


In [4]:
# List all blobs in the container
blob_list = list(container_client.list_blobs())

print(f"\nFound {len(blob_list)} files in the container:")
for idx, blob in enumerate(blob_list[:10], 1):
    size_mb = blob.size / (1024 * 1024)
    print(f"{idx}. {blob.name} ({size_mb:.2f} MB)")

if len(blob_list) > 10:
    print(f"... and {len(blob_list) - 10} more files")

INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://genaiesgsa.blob.core.windows.net/suzlon-documents?restype=REDACTED&comp=REDACTED'
Request method: 'GET'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.27.1 Python/3.12.12 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '08b06d47-fcf5-11f0-82ce-1418c33ebd62'
    'Authorization': 'REDACTED'
No body was attached to the request
INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/xml'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '9722ae7d-f01e-0086-0301-91b9f2000000'
    'x-ms-client-request-id': '08b06d47-fcf5-11f0-82ce-1418c33ebd62'
    'x-ms-version': 'REDACTED'
    'Date': 'Thu, 29 Jan 2026 09:29:39 GMT'



Found 54 files in the container:
1. 1. FY23-24 Business_Responsibility_and_Sustainability_Report.pdf (0.54 MB)
2. 1. Sustainability_Policy.pdf (0.11 MB)
3. 10. Energy_Management_Policy.pdf (2.68 MB)
4. 11. Sustainable_Sourcing_Policy.pdf (0.11 MB)
5. 12. Water_Stewardship_Policy.pdf (2.30 MB)
6. 13. DEIB_Policy.pdf (0.15 MB)
7. 14. Responsible_Business_Conduct_Policy.pdf (0.13 MB)
8. 15. Corporate_Governance_Policy.pdf (0.12 MB)
9. 16. Risk_Management_Policy.pdf (0.22 MB)
10. 17. Code_of_Ethics_for_Directors_and_Senior_Management.pdf (0.18 MB)
... and 44 more files


In [5]:
def download_blob(blob_name, download_dir):
    """
    Download a single blob from the container
    
    Args:
        blob_name: Name of the blob to download
        download_dir: Local directory to save the file
    
    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # Create blob client
        blob_client = blob_service_client.get_blob_client(
            container=CONFIG["container_name"], 
            blob=blob_name
        )
        
        # Create full local path (preserve folder structure if blob has path)
        local_file_path = Path(download_dir) / blob_name
        
        # Create parent directories if they don't exist
        local_file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Download the blob
        with open(local_file_path, "wb") as download_file:
            download_file.write(blob_client.download_blob().readall())
        
        return True
    except Exception as e:
        logger.error(f"Error downloading {blob_name}: {str(e)}")
        return False

print("Download function defined successfully!")

Download function defined successfully!


In [6]:
# Create download directory if it doesn't exist
download_path = Path(CONFIG["download_directory"])
download_path.mkdir(parents=True, exist_ok=True)

print(f"\nStarting download of {len(blob_list)} files...\n")

# Download all blobs with progress bar
successful_downloads = 0
failed_downloads = 0
failed_files = []

for blob in tqdm(blob_list, desc="Downloading files", unit="file"):
    if download_blob(blob.name, CONFIG["download_directory"]):
        successful_downloads += 1
    else:
        failed_downloads += 1
        failed_files.append(blob.name)

# Summary
print(f"\n{'='*60}")
print(f"Download Complete!")
print(f"{'='*60}")
print(f"Total files: {len(blob_list)}")
print(f"Successfully downloaded: {successful_downloads}")
print(f"Failed downloads: {failed_downloads}")
print(f"Download location: {download_path.absolute()}")

if failed_files:
    print(f"\nFailed files:")
    for file in failed_files:
        print(f"  - {file}")


Starting download of 54 files...



Request method: 'GET'
Request headers:
    'x-ms-range': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.27.1 Python/3.12.12 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '1095a1c0-fcf5-11f0-9a4c-1418c33ebd62'
    'Authorization': 'REDACTED'
No body was attached to the request
INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 206
Response headers:
    'Content-Length': '562572'
    'Content-Type': 'application/pdf'
    'Content-Range': 'REDACTED'
    'Last-Modified': 'Wed, 21 Jan 2026 11:41:39 GMT'
    'Accept-Ranges': 'REDACTED'
    'ETag': '"0x8DE58E20A66C7B8"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '9722c40f-f01e-0086-2901-91b9f2000000'
    'x-ms-client-request-id': '1095a1c0-fcf5-11f0-9a4c-1418c33ebd62'
    'x-ms-version': 'REDACTED'
    'x-ms-tag-count': 'REDACTED'
    'x-ms-creation-time': 'REDACTED'
 


Download Complete!
Total files: 54
Successfully downloaded: 54
Failed downloads: 0
Download location: c:\Users\BillaVenuGopalReddy\Desktop\Projects\Suzlon_practice_RAG\Suzlon_RAG\downloaded_documents


In [7]:
# Verify downloaded files
downloaded_files = list(download_path.rglob('*.*'))
total_size = sum(f.stat().st_size for f in downloaded_files if f.is_file())

print(f"\nVerification:")
print(f"Files on disk: {len(downloaded_files)}")
print(f"Total size: {total_size / (1024**2):.2f} MB")
print(f"\nFile types:")

# Count file types
file_types = {}
for file in downloaded_files:
    ext = file.suffix.lower()
    file_types[ext] = file_types.get(ext, 0) + 1

for ext, count in sorted(file_types.items()):
    print(f"  {ext or 'no extension'}: {count} file(s)")


Verification:
Files on disk: 54
Total size: 126.58 MB

File types:
  .pdf: 54 file(s)


In [2]:
import os
from dotenv import load_dotenv
# 1. Load Configuration
load_dotenv(".env")

def get_env_or_raise(key: str) -> str:
    val = os.getenv(key)
    if not val:
        raise ValueError(f"Missing required environment variable: {key}")
    return val

mongo_uri =  get_env_or_raise("AZURE_COSMOS_MONGO_URI")
database_name =  get_env_or_raise("AZURE_COSMOS_DATABASE_NAME")
chat_session_collection = get_env_or_raise("AZURE_COSMOS_CONTAINER_NAME_SESSION")
chat_logs_collection = get_env_or_raise("AZURE_COSMOS_CONTAINER_NAME_LOGS")


In [3]:
from pymongo import MongoClient

mongo_client = MongoClient(mongo_uri)
cosmos_db = mongo_client[database_name]
sessions_collection = cosmos_db[chat_session_collection]

def list_sessions():
    # Get sessions from Cosmos DB
    sessions = list(sessions_collection.find({}).sort("updated_at", -1).limit(50))
    
    # Convert to frontend format
    formatted_sessions = []
    for session in sessions:
        formatted_sessions.append({
            "id": session.get("session_id"),
            "title": session.get("title", "New Chat"),
            "messages": session.get("messages", []),
            "createdAt": session.get("created_at").isoformat() if "created_at" in session else None,
            "updatedAt": session.get("updated_at").isoformat() if "updated_at" in session else None,
            "user_id": session.get("user_id", "anonymous")
        })
    
    return {"sessions": formatted_sessions}

C:\Users\BillaVenuGopalReddy\AppData\Local\Temp\ipykernel_25140\2360482236.py:3: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  mongo_client = MongoClient(mongo_uri)


In [4]:
list_sessions()

{'sessions': [{'id': 'f96079ef-c208-421a-9aeb-e8d12d5ebb54',
   'title': 'Suzlon Governance Overview',
   'messages': [{'role': 'user', 'content': 'Hi', 'timestamp': 1772085033038},
    {'role': 'assistant',
     'content': 'Hello. I am a Corporate Intelligence Assistant focused on analyzing Suzlon Energy Limited’s official disclosures and policies.',
     'timestamp': 1772085034912,
     'sources': [],
     'images': []}],
   'createdAt': '2026-02-26T05:50:30.018000',
   'updatedAt': '2026-02-26T05:50:36.003000',
   'user_id': 'anonymous'},
  {'id': 'a2293c93-c325-4012-aa27-e915f14e23e9',
   'title': 'Manufacturing ESG Metrics Overview',
   'messages': [{'role': 'user',
     'content': 'what are the ESG metrics for manufactring?',
     'timestamp': 1771399007382},
    {'role': 'assistant',
     'content': 'Key ESG metrics for manufacturing at Suzlon include:\n\n- Reductions in energy requirements of products and services\n- Water withdrawal by source, water sources significantly affec

In [27]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv(".env")

def get_env_or_raise(key: str) -> str:
    val = os.getenv(key)
    if not val:
        raise ValueError(f"Missing required environment variable: {key}")
    return val

CONFIG = {
    "cosmos": {
        "mongo_uri": get_env_or_raise("AZURE_COSMOS_MONGO_URI"),
        "database_name": get_env_or_raise("AZURE_COSMOS_DATABASE_NAME"),
        "chat_session_collection": get_env_or_raise("AZURE_COSMOS_CONTAINER_NAME_SESSION"),
        "chat_logs_collection": get_env_or_raise("AZURE_COSMOS_CONTAINER_NAME_LOGS"),
    }
}

mongo_client = MongoClient(CONFIG["cosmos"]["mongo_uri"])
cosmos_db = mongo_client[CONFIG["cosmos"]["database_name"]]
sessions_collection = cosmos_db[CONFIG["cosmos"]["chat_session_collection"]]
logs_collection = cosmos_db[CONFIG["cosmos"]["chat_logs_collection"]]


def search_chat_history(
    user_id: str = None,
    search_text: str = None,   # partial text - searches both user questions AND assistant answers
    limit: int = 20
) -> list[dict]:
    """
    Search sessions by user_id and/or partial text in messages (questions or answers).
    Returns session_id and title of matching sessions.
    """

    filter_conditions = []

    # Filter by user_id if provided
    if user_id:
        filter_conditions.append({"user_id": user_id})

    # Search partial text across both user messages (questions) and assistant messages (answers)
    if search_text:
        filter_conditions.append({
            "messages": {
                "$elemMatch": {
                    "content": {"$regex": search_text, "$options": "i"}
                }
            }
        })

    # Build final filter
    if len(filter_conditions) == 0:
        mongo_filter = {}
    elif len(filter_conditions) == 1:
        mongo_filter = filter_conditions[0]
    else:
        mongo_filter = {"$and": filter_conditions}

    # Query sessions collection
    matched_sessions = list(
        sessions_collection.find(
            mongo_filter,
            {"session_id": 1, "title": 1, "_id": 0}
        )
        .sort("updated_at", -1)
        .limit(limit)
    )

    return [
        {
            "session_id": s.get("session_id"),
            "title": s.get("title", "New Chat")
        }
        for s in matched_sessions
    ]


# --- Usage ---
if __name__ == "__main__":
    results = search_chat_history(
        user_id="anonymous",       # optional - pass None to search all users
        search_text="leave policy for 2026",       # partial text - matches "leave policy", "annual leave", etc.
        limit=20
    )

    for r in results:
        print(r)
        # {"session_id": "054c2f1f-...", "title": "2026 Employee Leave Policy"}

C:\Users\BillaVenuGopalReddy\AppData\Local\Temp\ipykernel_44884\3336774693.py:22: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  mongo_client = MongoClient(CONFIG["cosmos"]["mongo_uri"])


{'session_id': '054c2f1f-2762-4237-9a00-1520578a50f5', 'title': '2026 Employee Leave Policy'}
{'session_id': '8f7e88a2-6756-4ef4-835f-41ad4d033fb7', 'title': 'Suzlon Leave Policy 2026'}


In [ ]:
import requests
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://esg-chatbot-wa01.azurewebsites.net"
# BASE_URL = "http://localhost:8003"
ENDPOINT = "/make_query"

failed_questions = [
    "What sustainability trainings are available for employees?",
    "What are the policies for circular economy or product recycling?",
    "How are sustainability efforts recognized internally?",
    "What certifications or recognitions support our excellence in sustainability?",
    "What is our renewable energy adoption target?",
    "What are the key social responsibility initiatives?",
    "What internal campaigns promote ESG awareness?",
    "How does the company monitor ESG compliance among suppliers?",
    "How do we manage energy, waste, and emissions reduction programs?"
]

def make_request(question):
    payload = {
        "history": [],
        "query": question,
        "user_id": "user_002",
        "chat_id": "chat_002"
    }

    start = time.time()
    try:
        response = requests.post(
            BASE_URL + ENDPOINT,
            json=payload,
            timeout=120
        )
        duration = round(time.time() - start, 2)
        print(response)
        return response.status_code, duration
    except Exception as e:
        duration = round(time.time() - start, 2)
        return str(e), duration


start_total = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(make_request, q) for q in esg_questions]
    
    for future in as_completed(futures):
        status, duration = future.result()
        print(f"Status: {status}, Time: {duration}s")

print("Total Wall Time:", round(time.time() - start_total, 2), "seconds")

<Response [200]>
Status: 200, Time: 2.69s
<Response [200]>
Status: 200, Time: 2.73s
<Response [200]>
Status: 200, Time: 7.27s
<Response [200]>
Status: 200, Time: 7.81s
<Response [200]>
Status: 200, Time: 8.29s
Total Wall Time: 8.3 seconds


In [1]:
import requests
import time
import json
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://esg-chatbot-wa01.azurewebsites.net"
# BASE_URL = "http://localhost:8003"
ENDPOINT = "/make_query"

failed_questions = [
    "What sustainability trainings are available for employees?",
    "What are the policies for circular economy or product recycling?",
    "How are sustainability efforts recognized internally?",
    "What certifications or recognitions support our excellence in sustainability?",
    "What is our renewable energy adoption target?",
    "What are the key social responsibility initiatives?",
    "What internal campaigns promote ESG awareness?",
    "How does the company monitor ESG compliance among suppliers?",
    "How do we manage energy, waste, and emissions reduction programs?"
]

results = []

def make_request(question):
    payload = {
        "history": [],
        "query": question,
        "user_id": "user_002",
        "chat_id": "chat_002"
    }

    start = time.time()
    timestamp = datetime.now().isoformat()
    try:
        response = requests.post(
            BASE_URL + ENDPOINT,
            json=payload,
            timeout=120
        )
        duration = round(time.time() - start, 2)

        try:
            response_body = response.json()
        except Exception:
            response_body = response.text

        return {
            "timestamp": timestamp,
            "question": question,
            "status_code": response.status_code,
            "duration_ms": duration * 1000,
            "response": response_body
        }

    except Exception as e:
        duration = round(time.time() - start, 2)
        return {
            "timestamp": timestamp,
            "question": question,
            "status_code": "ERROR",
            "duration_ms": duration * 1000,
            "response": str(e)
        }


start_total = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(make_request, q) for q in failed_questions]

    for i, future in enumerate(as_completed(futures), 1):
        result = future.result()
        results.append(result)

        # Pretty print each result as it completes
        print(f"\n{'='*60}")
        print(f"Request #{i} / {len(failed_questions)}")
        print(f"Timestamp  : {result['timestamp']}")
        print(f"Status     : {result['status_code']}")
        print(f"Time (ms)  : {result['duration_ms']}")
        print(f"Question   : {result['question']}")
        print(f"Response   : {result['response']}")
        print(f"{'='*60}")

total_time = round(time.time() - start_total, 2)
print(f"\nTotal Wall Time: {total_time} seconds")

# Save results to JSON file
output_file = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump({
        "total_wall_time_seconds": total_time,
        "total_requests": len(results),
        "results": results
    }, f, indent=4, ensure_ascii=False)

print(f"\nResults saved to: {output_file}")


Request #1 / 9
Timestamp  : 2026-02-27T11:03:16.653192
Status     : 200
Time (ms)  : 12240.0
Question   : What certifications or recognitions support our excellence in sustainability?
Response   : {'answer': 'Suzlon Energy Limited’s excellence in sustainability is supported by several external certifications and recognitions:\n\n- The company’s Occupational Health & Safety (OHS) Management System covers 100% of employees and is externally certified to the ISO 45001:2018 standard.\n- Suzlon adheres to Quality, Health, Safety, and Environment (QHSE) procedures, which are externally validated through ISO 45001:2018 certification.\n- Historically, Suzlon Group has also held certifications such as ISO 14001:2015 (Environmental Management) and OHSAS 18001:2007 (Occupational Health & Safety) across multiple business verticals.\n- It is a business requirement for all products and sites to be certified with ISO 9001:2015 (Quality Management), ISO 14001:2015, and OHSAS 18001:2007.\n\nThese cert